# Task 1: Third-order letter aproximation model

In [12]:
### IMPORTS
import os
from collections import defaultdict
import random

In [13]:
def remove_intro_and_outro(text):
    # Find the start marker for the actual book content
    start_marker = "*** START OF"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        # Find the next line break after the start marker and cut text after it
        text = text[start_idx + len(start_marker):].strip()

    # Find the end marker for the end of the book content
    end_marker = "*** END OF"
    end_idx = text.find(end_marker)
    if end_idx != -1:
        # Cut text before the end marker
        text = text[:end_idx].strip()

    return text

#### remove_intro_and_outro() 
the above method iterates through the text file provided, and removes all text before the section of the text with __"*** START OF"__ which corelates to the section of the text that covers the intro by Project Gutenberg placed in every written work.
In the same vain the method also checks for the __"*** END OF"__ section of text to remove all the endnotes and credits after the text, this leaves only the raw written text as it would be found in any book, without added watermarks and headers from Project Gutenberg 

In [14]:
def basic_cleaning(text):    
    cleaned_text = []
    for char in text:
        if char.isalpha() or char == ' ' or char == '.':
            cleaned_text.append(char)
    return ''.join(cleaned_text)

### basic_cleaning()
the above method iterates through the text file, this time inserting all text into an array of characters, we iterate through the array checking if any characters are empty spaces, periods, or uppercase, in the instance a character does not confrom to these conditions, the character is replaced with a null char.

In [15]:
def advanced_preprocess_text(file_path):
    # Step 1: Read the file
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Step 2: Remove intro and outro
    text = remove_intro_and_outro(text)
    
    # Step 3: Convert to uppercase
    text = text.upper()

    # Step 4: Remove unwanted characters (keep letters, spaces, and full stops)
    text = basic_cleaning(text)
    
    return text

#### advanced_preprocess_text()
finally this function calls all our previous methods to take in, iterate through, and clean through all the text, before outputting a file 

In [16]:
def trigram_iteration(text, trigram_counts):
    # Iterate through the text to count trigrams
    for i in range(len(text) - 2):
        trigram = text[i:i+3]
        trigram_counts[trigram] += 1

    return trigram_counts


#### trigram_iteration()
this function is called and iterates through the text file splitting the text at every three characters to create a trigram

In [17]:
# Combined output file paths
combined_processed_file = 'processedworks.txt'
combined_trigrams_file = 'trigrams.txt'

# Ensure existing files are cleared
with open(combined_processed_file, 'w', encoding='utf-8') as f:
    f.write('')
with open(combined_trigrams_file, 'w', encoding='utf-8') as f:
    f.write('')

# Initialize a global trigram dictionary
global_trigram_counts = defaultdict(int)


The above code does two things, the first is that it it initilaises a locaiton for the works to be saved to, as well as ensuring they are empty before write, and then initialises the dictionary that trigrams will be stored in 

In [18]:
# Find the base directory regardless of system
base_dir = os.path.join(os.getcwd(), 'englishworks')
if not os.path.exists(base_dir):
    raise FileNotFoundError(f"Directory '{base_dir}' does not exist")

for file in os.listdir(base_dir):
    file_path = os.path.join(base_dir, file)
    if os.path.isfile(file_path):
        processed_text = advanced_preprocess_text(file_path)

        # Append the processed text to the combined file
        with open(combined_processed_file, 'a', encoding='utf-8') as processed_file:
            processed_file.write(f'--- {file} ---\n')
            processed_file.write(processed_text + '\n\n')

        print(f"Processed and saved: {file}")

        # Update trigram counts
        global_trigram_counts = trigram_iteration(processed_text, global_trigram_counts)

# Save all 5 combined trigram counts to a single file
with open(combined_trigrams_file, 'w', encoding='utf-8') as trigram_file:
    for trigram, count in global_trigram_counts.items():
        trigram_file.write(f'{trigram}: {count}\n')

print(f"All processed texts saved in {combined_processed_file}")
print(f"All trigram counts saved in {combined_trigrams_file}")

Processed and saved: pg1342.txt
Processed and saved: frankenstein.txt
Processed and saved: dracula.txt
Processed and saved: mobydick.txt
Processed and saved: phantom.txt
All processed texts saved in processedworks.txt
All trigram counts saved in trigrams.txt


#### runner
the above code acts as a runner to all our above functions, it first uses the base library for __OS__ grabs the root directory, and loads the text files in the "englishworks" directory, once all these works are loaded, they are all merged and cleaned, and then outputted to a single file titled __"output.txt"__

# ===================================================================================
# Task 2: Third-order letter aproximation generation

In [ ]:
def get_possible_trigrams(trigram_model, last_two_chars):
    possible_trigrams = {}
    for trigram, count in trigram_model.items():
        if trigram.startswith(last_two_chars):
            possible_trigrams[trigram] = count
    return possible_trigrams

#### get_possible_trigrams()
The above function takes in the trigram model parameter as well as the last_two_chars parameter, which marks the two characters as found in the generated text, and search through the trigram model, to find all trigrams that begin with the last two chars parameter, giving all possible cmbinations of "next characters" which are then stored in a dictionary

In [ ]:
def weighted_random_choice(possible_trigrams):
    third_letters = []
    counts = []
    
    # Extract the third letters and counts from the possible trigrams
    for trigram, count in possible_trigrams.items():
        third_letters.append(trigram[2])
        counts.append(count)
    
    # Randomly select a third letter based on the counts
    if third_letters:  
        return random.choices(third_letters, weights=counts)[0]
    else:
        return None  # No possible trigrams found

#### weighted_random_choice()
The above code takes the dictionary made previosly and extracts the third letter from each trigram, it uses the counts given as weighting for random selection, for example with the two words _"THE"_ and _"THA"_ where _"THE"_ appears 150 times and _"THA"_ appears 75 times, then the probability using the third letter would look like:

    > E::150/X

    > A::75/X
